# 🚀 Qwen3-VL-8B + LoRA | RTX 5060 Ti 16GB + Windows
**Qwen3-VL-8B-Instruct + QLoRA Fine-tuning for VQA**

> ⚠️ **설치 순서 중요**: PyTorch → transformers(git) → peft/accelerate 순으로 설치
> ⚠️ **VRAM OOM 시**: `MODEL_ID`를 `Qwen/Qwen2.5-VL-7B-Instruct`로 변경


In [ ]:
# Cell 1-A: 기존 충돌 패키지 제거
!pip uninstall -y transformers peft accelerate bitsandbytes -q

In [ ]:
# Cell 1-B: PyTorch 설치 (CUDA 12.8)
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [ ]:
# Cell 1-C: transformers 최신 (Qwen3-VL 필수: >=4.57.0, git 소스 설치)
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
# Cell 1-D: 나머지 패키지 (transformers 설치 후에!)
!pip install "accelerate>=1.0.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" pillow pandas -q --upgrade

In [ ]:
# Cell 2: 버전 및 GPU 확인
import torch, transformers, peft, accelerate, bitsandbytes
print(f"PyTorch:        {torch.__version__}")
print(f"transformers:   {transformers.__version__}")
print(f"peft:           {peft.__version__}")
print(f"accelerate:     {accelerate.__version__}")
print(f"bitsandbytes:   {bitsandbytes.__version__}")
print(f"CUDA:           {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name()}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 3: 임포트 + CUDA 최적화
import os, re, math, random, gc
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None

# RTX 5060 Ti CUDA 최적화
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False   # Windows 안정성
torch.backends.cuda.enable_flash_sdp(True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# Cell 4: 하이퍼파라미터
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

IMAGE_SIZE = 512
MAX_NEW_TOKENS = 4
SEED = 42

NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8         # effective batch = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.1
PATIENCE = 2

INFERENCE_BATCH = 2
NUM_WORKERS = 0        # ★ Windows: 반드시 0

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print(f"학습: {len(train_df)}개, 테스트: {len(test_df)}개")

In [ ]:
# Cell 5: 모델 로드 + LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",   # ★ Windows: sdpa 안정적
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"VRAM 사용: {torch.cuda.memory_allocated()/1024**3:.1f} GB / 16.0 GB")

In [ ]:
# Cell 6: 프롬프트
SYSTEM_INSTRUCT = (
    "You are an expert visual question answering system. "
    "Look at the image carefully, analyze the question, and select the single best answer. "
    "You must respond with ONLY one lowercase letter: a, b, c, or d. "
    "Do not include any explanation, punctuation, or extra text."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n\n"
        f"Options:\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Based on the image, the correct answer is:"
    )

In [ ]:
# Cell 7: 데이터셋 + 콜레이터 (Label Masking)
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        q, a, b, c, d = str(row["question"]), str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, prompt_texts = [], [], []
        for sample in batch:
            msgs, img = sample["messages"], sample["image"]

            full_text = self.processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
            texts.append(full_text)
            images.append(img)

            if self.train:
                prompt_text = self.processor.apply_chat_template(
                    msgs[:-1], tokenize=False, add_generation_prompt=True
                )
                prompt_texts.append(prompt_text)

        enc = self.processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            for idx, pt in enumerate(prompt_texts):
                prompt_enc = self.processor.tokenizer(
                    pt, return_tensors="pt", add_special_tokens=False
                )
                prompt_len = prompt_enc["input_ids"].shape[1]
                labels[idx, :prompt_len] = -100
            if self.processor.tokenizer.pad_token_id is not None:
                labels[labels == self.processor.tokenizer.pad_token_id] = -100
            enc["labels"] = labels

        return enc

In [ ]:
# Cell 8: 데이터 분리 + DataLoader
train_df_shuffled = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split = int(len(train_df_shuffled) * 0.9)
train_subset = train_df_shuffled.iloc[:split]
valid_subset = train_df_shuffled.iloc[split:]
print(f"학습: {len(train_subset)}개, 검증: {len(valid_subset)}개")

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# ★ Windows: num_workers=0, persistent_workers 없음
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS, pin_memory=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS, pin_memory=True,
)

In [ ]:
# Cell 9: 학습 루프
# ★ model.to(device) 없음 — device_map="auto"가 이미 GPU 배치

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
scaler = torch.amp.GradScaler("cuda", enabled=True)

best_val_loss = float("inf")
patience_counter = 0
SAVE_DIR = "qwen3_vl_8b_lora_best"

print(f"총 학습 스텝: {num_training_steps}, 워밍업: {num_warmup_steps}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

# ★ CUDA 워밍업 (Windows 첫 배치 프리징 방지)
print("CUDA 워밍업 중...")
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
    wb = next(iter(train_loader))
    wb = {k: v.to(device, non_blocking=True) for k, v in wb.items()}
    _ = model(**wb)
    del wb
    torch.cuda.empty_cache()
print("워밍업 완료!\n")

global_step = 0
for epoch in range(NUM_EPOCHS):
    model.train()
    running, epoch_loss, epoch_steps = 0.0, 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            epoch_loss += avg_loss
            epoch_steps += 1
            pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
            running = 0.0

    # 남은 gradient flush
    if step % GRAD_ACCUM != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

    # Validation
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]", unit="batch"):
            vb = {k: v.to(device, non_blocking=True) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    avg_val = val_loss / max(val_steps, 1)
    avg_train = epoch_loss / max(epoch_steps, 1)
    peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    print(f"\n[Epoch {epoch+1}] train: {avg_train:.4f} | valid: {avg_val:.4f} | peak VRAM: {peak:.1f}GB")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_counter = 0
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  -> Best model saved! (val_loss: {best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("  -> Early stopping!")
            break

    gc.collect()
    torch.cuda.empty_cache()

# Best 모델 로드
print(f"\nBest val loss: {best_val_loss:.4f}")
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, SAVE_DIR)
model.eval()
print("Best model loaded!")

In [ ]:
# Cell 10: 추론
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a","b","c","d"]:
        return last
    for pat in [r'\b([abcd])\b', r'\(([abcd])\)', r'answer\s*(?:is|:)\s*([abcd])']:
        m = re.search(pat, text)
        if m:
            return m.group(1)
    return "a"


def batch_inference(model, processor, test_df, batch_size=2):
    model.eval()
    preds = []
    for start in tqdm(range(0, len(test_df), batch_size), desc="Inference", unit="batch"):
        end = min(start + batch_size, len(test_df))
        rows = test_df.iloc[start:end]
        texts, images = [], []

        for _, row in rows.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            msgs = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            text = processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
            texts.append(text)
            images.append(img)

        inputs = processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        ).to(device)

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        # 생성된 토큰만 추출
        input_len = inputs["input_ids"].shape[1]
        for i_b in range(len(rows)):
            gen_ids = out_ids[i_b][input_len:]
            txt = processor.decode(gen_ids, skip_special_tokens=True)
            preds.append(extract_choice(txt))

        del inputs, out_ids
        torch.cuda.empty_cache()

    return preds


preds = batch_inference(model, processor, test_df, batch_size=INFERENCE_BATCH)

os.makedirs("content", exist_ok=True)
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("content/submission.csv", index=False)
print(f"\nSaved content/submission.csv ({len(submission)}개)")
print(submission["answer"].value_counts().sort_index())